# 03 - Model Architecture: Whisper Encoder + Audio Projector

**Goal**: Initialize a Qwen2-VL model with audio capability — load base weights + Whisper encoder weights, verify forward pass, push to HuggingFace.

**What we do here**:
1. Load Qwen2-VL-7B-Instruct config and extend it with `Qwen2VLAudioConfig`
2. Create the model from modified config (audio components start random)
3. Load base Qwen2-VL-7B weights with `strict=False` (only audio keys missing)
4. Copy Whisper-large-v3-turbo encoder weights into `model.model.audio_encoder`
5. Verify model structure and run a test forward pass with audio input
6. Save and push the complete model to HuggingFace

**Prerequisites**:
- Notebook 02 completed (processor with audio tokens pushed to `DanJZY/Qwen2-VL-7B-Speech`)
- Transformers fork with audio encoder/projector committed (`ZhuoyuanJiang/transformers` branch `speech-qwen2vl`)

**Where to run**: Colab Pro (L4 24GB or A100 40GB+) or server with A6000 48GB.
Peak GPU memory: ~18.5GB.

## 1. Environment Setup

In [1]:
# Install dependencies, then our fork LAST to prevent overwrites.
# --force-reinstall --no-deps ensures we always get our fork code
# (pip caches git installs and won't re-download after runtime restart).
# Pinned to exact commit hashes for reproducibility.
#
# On server with editable installs already set up, skip this cell.

!pip install -q datasets librosa soundfile huggingface_hub accelerate
!pip install -q "tokenizers>=0.21,<0.22"
!pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q --force-reinstall --no-deps git+https://github.com/ZhuoyuanJiang/transformers.git@934129b7701e7607facb39f286afc6bc4cc657df
!pip install -q --force-reinstall --no-deps git+https://github.com/ZhuoyuanJiang/Qwen3-VL.git@56b0756a768cc3b01cba45b01c1bc3c8cb74ea3f#subdirectory=qwen-vl-utils

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
import torch
import numpy as np
from io import BytesIO
from datasets import load_dataset
from huggingface_hub import HfApi, login, get_token
from transformers import (
    AutoConfig,
    Qwen2VLForConditionalGeneration,
    Qwen2VLProcessor,
    WhisperForConditionalGeneration,
)
from transformers.models.qwen2_vl.configuration_qwen2_vl import Qwen2VLAudioConfig
import transformers

print(f"transformers: {transformers.__version__}")
print(f"transformers path: {transformers.__file__}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

transformers: 4.56.0.dev0
transformers path: /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
torch: 2.4.1+cu121
CUDA available: True
GPU: NVIDIA L4
GPU memory: 22.0 GB


In [3]:
# HuggingFace login — needed to push model to HF
HF_TOKEN = None

# Check for cached token (from a previous login() call)
HF_TOKEN = get_token()

# Try Colab Secrets
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

# Try environment variable (server / local machine)
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")
else:
    print("No cached token found. Please paste your token below (only needed once):")
    login()

Logged in to HuggingFace.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 2. Load & Modify Config

Load the base Qwen2-VL-7B-Instruct config, then extend it with audio support:
- Add `Qwen2VLAudioConfig()` with Whisper-large-v3-turbo dimensions
- Set `audio_token_id=151658` (`<|audio_pad|>` from Notebook 02)

In [4]:
# Load base config from Qwen2-VL-7B-Instruct
config = AutoConfig.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

print(f"Model type: {config.model_type}")
print(f"Text hidden size: {config.text_config.hidden_size}")
print(f"Vision hidden size: {config.vision_config.hidden_size}")
print(f"image_token_id: {config.image_token_id}")
print(f"video_token_id: {config.video_token_id}")
print(f"audio_config before: {getattr(config, 'audio_config', None)}")
print(f"audio_token_id before: {getattr(config, 'audio_token_id', None)}")

Model type: qwen2_vl
Text hidden size: 3584
Vision hidden size: 3584
image_token_id: 151655
video_token_id: 151656
audio_config before: None
audio_token_id before: None


In [5]:
# Add audio config with Whisper-large-v3-turbo dimensions
config.audio_config = Qwen2VLAudioConfig(
    d_model=1280,
    encoder_layers=32,
    encoder_attention_heads=20,
    encoder_ffn_dim=5120,
    num_mel_bins=128,
    max_source_positions=1500,
)
config.audio_token_id = 151658  # <|audio_pad|> from Notebook 02

print(f"audio_config: {config.audio_config}")
print(f"audio_token_id: {config.audio_token_id}")
print(f"\nAudio config details:")
print(f"  d_model: {config.audio_config.d_model}")
print(f"  encoder_layers: {config.audio_config.encoder_layers}")
print(f"  num_mel_bins: {config.audio_config.num_mel_bins}")
print(f"  max_source_positions: {config.audio_config.max_source_positions}")

audio_config: Qwen2VLAudioConfig {
  "activation_function": "gelu",
  "d_model": 1280,
  "encoder_attention_heads": 20,
  "encoder_ffn_dim": 5120,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 32,
  "max_source_positions": 1500,
  "model_type": "qwen2_vl",
  "num_mel_bins": 128,
  "scale_embedding": false,
  "transformers_version": "4.56.0.dev0"
}

audio_token_id: 151658

Audio config details:
  d_model: 1280
  encoder_layers: 32
  num_mel_bins: 128
  max_source_positions: 1500


### Config round-trip test

Verify that saving and loading the config preserves the audio fields.

In [6]:
# Save config to temp dir, reload, verify audio fields survive
import tempfile, json

with tempfile.TemporaryDirectory() as tmpdir:
    config.save_pretrained(tmpdir)

    # Check the saved JSON
    with open(os.path.join(tmpdir, "config.json")) as f:
        saved = json.load(f)
    assert "audio_config" in saved, "audio_config missing from saved config!"
    assert saved["audio_token_id"] == 151658, "audio_token_id wrong in saved config!"
    print(f"Saved audio_config keys: {list(saved['audio_config'].keys())}")

    # Reload and verify
    config_reloaded = AutoConfig.from_pretrained(tmpdir)
    assert isinstance(config_reloaded.audio_config, Qwen2VLAudioConfig), \
        f"audio_config deserialized as {type(config_reloaded.audio_config)}, expected Qwen2VLAudioConfig"
    assert config_reloaded.audio_config.d_model == 1280
    assert config_reloaded.audio_token_id == 151658
    print("Config round-trip verified: audio_config deserializes correctly.")

Saved audio_config keys: ['_name_or_path', 'activation_function', 'add_cross_attention', 'architectures', 'bad_words_ids', 'begin_suppress_tokens', 'bos_token_id', 'chunk_size_feed_forward', 'cross_attention_hidden_size', 'd_model', 'decoder_start_token_id', 'diversity_penalty', 'do_sample', 'early_stopping', 'encoder_attention_heads', 'encoder_ffn_dim', 'encoder_layerdrop', 'encoder_layers', 'encoder_no_repeat_ngram_size', 'eos_token_id', 'exponential_decay_length_penalty', 'finetuning_task', 'forced_bos_token_id', 'forced_eos_token_id', 'id2label', 'is_decoder', 'is_encoder_decoder', 'label2id', 'length_penalty', 'max_length', 'max_source_positions', 'min_length', 'model_type', 'no_repeat_ngram_size', 'num_beam_groups', 'num_beams', 'num_mel_bins', 'num_return_sequences', 'output_attentions', 'output_hidden_states', 'output_scores', 'pad_token_id', 'prefix', 'problem_type', 'pruned_heads', 'remove_invalid_values', 'repetition_penalty', 'return_dict', 'return_dict_in_generate', 'scale

## 3. Create Model & Load Base Weights

Create the model from our modified config. At this point:
- `visual`, `language_model`, `lm_head` — random init
- `audio_encoder`, `audio_projector` — random init

Then load the pretrained Qwen2-VL-7B-Instruct weights with `strict=False`. The only missing keys should be the audio components.

In [7]:
# Load model with our modified config in bf16
# This creates the full model (including audio_encoder/audio_projector) with random weights,
# then loads Qwen2-VL-7B-Instruct pretrained weights for everything except audio components.
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    config=config,
    torch_dtype=torch.bfloat16,
    device_map="cpu",  # load to CPU first, move to GPU later
    ignore_mismatched_sizes=False,
)

print(f"Model type: {type(model).__name__}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen2VLForConditionalGeneration were not initialized from the model checkpoint at Qwen/Qwen2-VL-7B-Instruct and are newly initialized: ['model.audio_encoder.conv1.bias', 'model.audio_encoder.conv1.weight', 'model.audio_encoder.conv2.bias', 'model.audio_encoder.conv2.weight', 'model.audio_encoder.embed_positions.weight', 'model.audio_encoder.layer_norm.bias', 'model.audio_encoder.layer_norm.weight', 'model.audio_encoder.layers.0.fc1.bias', 'model.audio_encoder.layers.0.fc1.weight', 'model.audio_encoder.layers.0.fc2.bias', 'model.audio_encoder.layers.0.fc2.weight', 'model.audio_encoder.layers.0.final_layer_norm.bias', 'model.audio_encoder.layers.0.final_layer_norm.weight', 'model.audio_encoder.layers.0.self_attn.k_proj.weight', 'model.audio_encoder.layers.0.self_attn.out_proj.bias', 'model.audio_encoder.layers.0.self_attn.out_proj.weight', 'model.audio_encoder.layers.0.self_attn.q_proj.bias', 'model.audio_encoder.layers.0.self_attn.q_proj.weight', 'model.audio_encoder.lay

Model type: Qwen2VLForConditionalGeneration
Device: cpu
Dtype: torch.bfloat16


In [8]:
# Verify audio components exist
assert hasattr(model.model, 'audio_encoder'), "audio_encoder not found!"
assert hasattr(model.model, 'audio_projector'), "audio_projector not found!"
print(f"audio_encoder type: {type(model.model.audio_encoder).__name__}")
print(f"audio_projector type: {type(model.model.audio_projector).__name__}")
print("Audio components created successfully.")

audio_encoder type: WhisperEncoder
audio_projector type: Sequential
Audio components created successfully.


## 4. Load Whisper Encoder Weights

Load whisper-large-v3-turbo and copy its encoder state_dict into our model's `audio_encoder`.

**Important**: Whisper must load in float32 to avoid dtype mismatch errors during its own init.
We convert to bf16 after copying weights.

In [9]:
# Load Whisper in float32 (required — loading in bf16 causes internal dtype mismatches)
whisper = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3-turbo",
    torch_dtype=torch.float32,
)
print(f"Whisper loaded: {type(whisper).__name__}")
print(f"Whisper encoder layers: {len(whisper.model.encoder.layers)}")
print(f"Whisper encoder dtype: {whisper.model.encoder.dtype}")

Whisper loaded: WhisperForConditionalGeneration
Whisper encoder layers: 32
Whisper encoder dtype: torch.float32


In [10]:
# Copy Whisper encoder weights into our model's audio_encoder
whisper_encoder_state = whisper.model.encoder.state_dict()
load_result = model.model.audio_encoder.load_state_dict(whisper_encoder_state, strict=True)

print(f"Missing keys: {load_result.missing_keys}")
print(f"Unexpected keys: {load_result.unexpected_keys}")
assert len(load_result.missing_keys) == 0, f"Missing keys: {load_result.missing_keys}"
assert len(load_result.unexpected_keys) == 0, f"Unexpected keys: {load_result.unexpected_keys}"
print("Whisper encoder weights loaded with 0 missing, 0 unexpected keys.")

Missing keys: []
Unexpected keys: []
Whisper encoder weights loaded with 0 missing, 0 unexpected keys.


In [11]:
# Delete Whisper model to free memory (~3.2GB)
del whisper, whisper_encoder_state
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Whisper model deleted. Memory freed.")

Whisper model deleted. Memory freed.


## 5. Verify Model Structure

Check the model architecture and count parameters per component.

In [12]:
# Count parameters per component
def count_params(module):
    return sum(p.numel() for p in module.parameters())

total = count_params(model)
visual = count_params(model.model.visual)
language = count_params(model.model.language_model)
lm_head = count_params(model.lm_head)
audio_enc = count_params(model.model.audio_encoder)
audio_proj = count_params(model.model.audio_projector)

print(f"{'Component':<25} {'Params':>15} {'Size (bf16)':>12}")
print(f"{'-'*25} {'-'*15} {'-'*12}")
print(f"{'visual (ViT)':<25} {visual:>15,} {visual * 2 / 1024**3:>10.2f} GB")
print(f"{'language_model (LLM)':<25} {language:>15,} {language * 2 / 1024**3:>10.2f} GB")
print(f"{'lm_head':<25} {lm_head:>15,} {lm_head * 2 / 1024**3:>10.2f} GB")
print(f"{'audio_encoder (Whisper)':<25} {audio_enc:>15,} {audio_enc * 2 / 1024**3:>10.2f} GB")
print(f"{'audio_projector (MLP)':<25} {audio_proj:>15,} {audio_proj * 2 / 1024**3:>10.2f} GB")
print(f"{'-'*25} {'-'*15} {'-'*12}")
print(f"{'TOTAL':<25} {total:>15,} {total * 2 / 1024**3:>10.2f} GB")

Component                          Params  Size (bf16)
------------------------- --------------- ------------
visual (ViT)                  675,759,104       1.26 GB
language_model (LLM)        7,070,619,136      13.17 GB
lm_head                       544,997,376       1.02 GB
audio_encoder (Whisper)       636,968,960       1.19 GB
audio_projector (MLP)          17,439,744       0.03 GB
------------------------- --------------- ------------
TOTAL                       8,945,784,320      16.66 GB


In [13]:
# Print weight initialization status
print("Weight initialization status:")
print(f"  visual (ViT):           Qwen2-VL-7B-Instruct (pretrained)")
print(f"  language_model (LLM):   Qwen2-VL-7B-Instruct (pretrained)")
print(f"  lm_head:                Qwen2-VL-7B-Instruct (pretrained)")
print(f"  audio_encoder:          whisper-large-v3-turbo (pretrained)")
print(f"  audio_projector:        Random init (will be trained in Session 5)")

# Quick sanity check: audio_projector should have non-zero weights (random init)
# and audio_encoder conv1 should match Whisper's known pattern
proj_weight = model.model.audio_projector[0].weight
enc_conv1 = model.model.audio_encoder.conv1.weight
print(f"\naudio_projector[0].weight: mean={proj_weight.float().mean():.6f}, std={proj_weight.float().std():.6f}")
print(f"audio_encoder.conv1.weight: mean={enc_conv1.float().mean():.6f}, std={enc_conv1.float().std():.6f}")

Weight initialization status:
  visual (ViT):           Qwen2-VL-7B-Instruct (pretrained)
  language_model (LLM):   Qwen2-VL-7B-Instruct (pretrained)
  lm_head:                Qwen2-VL-7B-Instruct (pretrained)
  audio_encoder:          whisper-large-v3-turbo (pretrained)
  audio_projector:        Random init (will be trained in Session 5)

audio_projector[0].weight: mean=0.000004, std=0.019877
audio_encoder.conv1.weight: mean=0.000018, std=0.017555


## 6. Test Forward Pass

Load the processor from Notebook 02, prepare a real audio sample, and run a forward pass through the model.

In [14]:
# Load processor from HuggingFace (pushed in Notebook 02)
from qwen_vl_utils import process_vision_info

REPO_ID = "DanJZY/Qwen2-VL-7B-Speech"
processor = Qwen2VLProcessor.from_pretrained(REPO_ID)
print(f"Processor loaded from {REPO_ID}")
print(f"audio_pad token ID: {processor.tokenizer.convert_tokens_to_ids('<|audio_pad|>')}")

Processor loaded from DanJZY/Qwen2-VL-7B-Speech
audio_pad token ID: 151658


In [15]:
# Load a test audio sample
ds_stream = load_dataset(
    "speechbrain/LargeScaleASR",
    data_files="small/train-00000*",
    streaming=True,
    split="train",
)
sample = next(iter(ds_stream))
print(f"Sample text: {sample['text'][:80]}...")
print(f"Sample duration: {sample['duration']:.2f}s")

Sample text: AND WHAT ABOUT INTEROPERABILITY IN THE RAIL SECTOR ARE NATIONAL BARRIERS PREVENT...
Sample duration: 17.12s


In [16]:
# Prepare input through the processor pipeline
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": sample["wav"]["bytes"]},
            {"type": "text", "text": "Transcribe this audio."},
        ],
    },
]

image_inputs, video_inputs, audio_inputs = process_vision_info(messages)
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

batch = processor(
    text=[text],
    audios=audio_inputs,
    return_tensors="pt",
    padding=True,
)

print("Batch contents:")
for key, value in batch.items():
    if hasattr(value, 'shape'):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"  {key}: {value}")

Batch contents:
  input_ids: shape=torch.Size([1, 882]), dtype=torch.int64
  attention_mask: shape=torch.Size([1, 882]), dtype=torch.int64
  audio_features: shape=torch.Size([1, 128, 3000]), dtype=torch.float32
  audio_lengths: shape=torch.Size([1]), dtype=torch.int64


In [17]:
# Move model and batch to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

if torch.cuda.is_available():
    print(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

GPU memory after model load: 16.71 GB


In [18]:
# Forward pass
model.eval()
with torch.no_grad():
    outputs = model(**batch)

print(f"Output type: {type(outputs).__name__}")
print(f"Logits shape: {outputs.logits.shape}")
print(f"Logits dtype: {outputs.logits.dtype}")
print(f"Expected: (1, {batch['input_ids'].shape[1]}, {config.text_config.vocab_size})")

assert outputs.logits.shape[0] == 1, "Batch size mismatch"
assert outputs.logits.shape[1] == batch['input_ids'].shape[1], "Sequence length mismatch"
assert outputs.logits.shape[2] == config.text_config.vocab_size, "Vocab size mismatch"
assert not torch.isnan(outputs.logits).any(), "NaN in logits!"
assert not torch.isinf(outputs.logits).any(), "Inf in logits!"
print("\nForward pass verified: logits shape correct, no NaN/Inf.")

if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

Output type: Qwen2VLCausalLMOutputWithPast
Logits shape: torch.Size([1, 882, 152064])
Logits dtype: torch.bfloat16
Expected: (1, 882, 152064)

Forward pass verified: logits shape correct, no NaN/Inf.
Peak GPU memory: 17.39 GB


## 7. Save & Push to HuggingFace

Save the complete model (base weights + Whisper encoder + random projector) and push to HuggingFace.

We also verify the save/load round-trip: reload the model and check that audio weights survived.

In [19]:
# Move model back to CPU for saving (reduces GPU memory during upload)
model = model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Save locally
SAVE_DIR = "./Qwen2-VL-7B-Speech"
model.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}/")
print(f"Files: {os.listdir(SAVE_DIR)}")

Model saved to ./Qwen2-VL-7B-Speech/
Files: ['model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors', 'model-00001-of-00004.safetensors', 'config.json', 'generation_config.json', 'model.safetensors.index.json', 'model-00002-of-00004.safetensors']


In [20]:
# Verify save/load round-trip: check audio weights survive
# Store a reference weight before reload
ref_conv1 = model.model.audio_encoder.conv1.weight.clone()
ref_proj = model.model.audio_projector[0].weight.clone()

# Reload from saved path
model_reloaded = Qwen2VLForConditionalGeneration.from_pretrained(
    SAVE_DIR,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
)

# Verify audio components exist and weights match
assert hasattr(model_reloaded.model, 'audio_encoder'), "audio_encoder missing after reload!"
assert hasattr(model_reloaded.model, 'audio_projector'), "audio_projector missing after reload!"

reloaded_conv1 = model_reloaded.model.audio_encoder.conv1.weight
reloaded_proj = model_reloaded.model.audio_projector[0].weight

assert torch.equal(ref_conv1, reloaded_conv1), "audio_encoder weights changed after save/load!"
assert torch.equal(ref_proj, reloaded_proj), "audio_projector weights changed after save/load!"
print("Save/load round-trip verified: audio weights match exactly.")
print(f"  audio_encoder.conv1: {torch.equal(ref_conv1, reloaded_conv1)}")
print(f"  audio_projector[0]: {torch.equal(ref_proj, reloaded_proj)}")

del model_reloaded
gc.collect()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Save/load round-trip verified: audio weights match exactly.
  audio_encoder.conv1: True
  audio_projector[0]: True


30970

In [21]:
# Push to HuggingFace
api = HfApi()
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=REPO_ID,
)
print(f"Model uploaded to https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00004.safetensors:   1%|          | 33.5MB / 4.97GB            

  ...0004-of-00004.safetensors:   1%|1         | 33.4MB / 3.00GB            

  ...0002-of-00004.safetensors:   1%|          | 25.1MB / 4.99GB            

  ...0003-of-00004.safetensors:   1%|          | 41.9MB / 4.93GB            

Model uploaded to https://huggingface.co/DanJZY/Qwen2-VL-7B-Speech


## 8. Cleanup

In [22]:
del model, batch, outputs, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Cleanup complete.")

GPU memory after cleanup: 0.01 GB
Cleanup complete.


## Summary

**What we built**:
1. Extended Qwen2-VL-7B config with `Qwen2VLAudioConfig` (Whisper dimensions) + `audio_token_id=151658`
2. Created model with audio encoder (WhisperEncoder, ~635M params) and audio projector (2-layer MLP, ~17M params)
3. Loaded pretrained weights: Qwen2-VL-7B-Instruct (visual + LLM + lm_head) + whisper-large-v3-turbo (encoder)
4. Verified forward pass with real audio input — logits produced correctly
5. Pushed complete model to HuggingFace (`DanJZY/Qwen2-VL-7B-Speech`)

**Weight status**:

| Component | Source | Status |
|-----------|--------|--------|
| visual (ViT) | Qwen2-VL-7B-Instruct | Pretrained |
| language_model (LLM) | Qwen2-VL-7B-Instruct | Pretrained |
| lm_head | Qwen2-VL-7B-Instruct | Pretrained |
| audio_encoder | whisper-large-v3-turbo | Pretrained |
| audio_projector | Random init | Untrained (Session 5) |

**Next**: Notebook 04 — Data Pipeline (build training dataset with audio features)